In [0]:
#adding src path and reading config file
import sys
sys.path.append("/Workspace/DataStore/DataStore/src")
import yaml
from core.Base1 import BaseConfig
from core.Read_Yaml import ReadYaml
from core.Create_Tabls import CreateTable
from core.Create_Schem import CreateSchema
from core.TableHandler import TableHandler
from pyspark.sql.functions import current_timestamp, col, lit, row_number
from pyspark.sql.window import Window
from utils.Utilss import Utils
from delta.tables import DeltaTable
sys.path.append("/Workspace/DataStore/DataStore/src")
path = "/Workspace/DataStore/DataStore/configs/gold.yaml"
settings_path = "/Workspace/DataStore/DataStore/configs/settings.yaml"



In [0]:
#creating schema if not exists
table_info= ReadYaml.read_yaml(path, "Queue")
table_info = BaseConfig(table_info)
CreateSchema.create_schema(spark,table_info.catalog, table_info.schema)

In [0]:
%sql
--creating a managed user silver table
CREATE TABLE IF NOT EXISTS datastore.gold.Queue
(
    Id              INT NOT NULL PRIMARY KEY,
    ExternalId      STRING NOT NULL,
    QueueName        STRING,
    BusinessunitId  STRING,
    IsActive        BOOLEAN,
    CreatedAt       TIMESTAMP,
    UpdatedAt      TIMESTAMP
)USING DELTA;

In [0]:
#generating dbId
res = spark.sql(f"select max(Id) as MaxId from {table_info.catalog}.{table_info.schema}.{table_info.tableName}").collect()[0] or 0

maxId = res["MaxId"] or 0
window = Window.partitionBy("CreatedAt").orderBy(lit(1))

In [0]:
#getting source siver layer data
source_df = (
    spark.table(f"{table_info.catalog}.silver.{table_info.tableName}")
    .withColumn("Id", row_number().over(window)) 
    .withColumn("IsActive", lit(True))
    .select(
        col("Id"),
        col("ExternalId"),
        col("QueueName"),
        col("BusinessunitId"),
        col("IsActive"),
        col("CreatedAt"),
        col("UpdatedAt")
    )
).filter(col("IsUpdated") == True)

source_df = source_df.drop("IsUpdated")



In [0]:
Utils.scdType2(spark, source_df, f"{table_info.catalog}.{table_info.schema}.{table_info.tableName}", ["ExternalId"])

In [0]:
#marking processed records as processed
silver_table = DeltaTable.forName(spark, f"{table_info.catalog}.silver.{table_info.tableName}")
silver_table.update(
    condition = col("IsUpdated") == True,
    set = {"IsUpdated": lit(False)}
)

In [0]:
%sql
select * from datastore.gold.Queue